# Lab 02-03 — BGE vs E5: head-to-head retrieval comparison

**Track 02 · Embeddings** — the model choice that bounds retrieval quality. The embedding model is the block that decides *which* passages a query can ever reach: retrieval quality is bounded by embedding quality before any retriever, prompt, or LLM gets a vote. This lab runs two local bi-encoder models head to head on the same corpus and the same 60 real Q/A pairs:

* **BGE** — `BAAI/bge-base-en-v1.5`. bge models are *trained* for cosine similarity: embeddings are produced with `normalize_embeddings=True`, so every vector leaves the model unit-length.
* **E5** — `intfloat/multilingual-e5-base`. E5 models are trained with instruction prefixes: the wrapper prepends `"query: "` to questions and `"passage: "` to passages automatically. Unlike BGE, the E5 wrapper does **not** normalize the output vectors.

Two caveats the lab makes visible:

1. **Prefixes matter.** `multilingual-e5-base` was trained on `query: ...` / `passage: ...` pairs. Embedding a question without the `query: ` prefix would put it in a different region of the space than the prefixed passages — the wrapper's auto-prefixing is exactly why it is used here.
2. **Normalization matters for fair cosine.** Cosine similarity is scale-invariant, so raw vs normalized vectors give identical *rankings* — but a raw E5 vector is several times longer than a unit vector, and raw dot products are not comparable to BGE's normalized ones. For a fair comparison (and to report mean/median similarity on the same scale), both models' matrices are L2-normalized here with numpy before scoring.

This notebook is **self-contained**: it imports LangChain, numpy, and pandas directly — no repo component library. The two embedders are built right here as small inline wrappers over `HuggingFaceEmbeddings` (BGE with `normalize_embeddings=True`, E5 with its `"query: "` / `"passage: "` instruction prefixes), which is exactly how the shared `src/embeddings/bge.py` and `src/embeddings/e5.py` components work underneath.

Metric: **answer-containment recall@1** — the fraction of questions whose gold answer string (case-insensitive, whitespace-stripped) appears inside the single most-similar passage. No LLM, no judge, no API: pure retrieval measurement on real Q/A pairs.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-huggingface`, `numpy`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   sentence-transformers -> local BGE + E5 embeddings
#   langchain-huggingface -> HuggingFaceEmbeddings (the universal embedder class)
#   pandas                -> reads the passages/test parquet corpus
#   numpy                 -> L2 normalization + cosine scoring matrices
%pip install -q sentence-transformers langchain-huggingface pandas numpy


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

# Silence the "Loading weights" progress bar (transformers honors
# HF_HUB_DISABLE_PROGRESS_BARS, read at huggingface_hub import time — set it
# before any third-party import).
import os

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import time
from pathlib import Path

import numpy as np
import pandas as pd

# LangChain + numpy/pandas — the only libraries this notebook needs.
# Nothing is imported from the repo's src/ component library.
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402


class BGEEmbedding(Embeddings):
    """Inline BGE wrapper — mirrors src/embeddings/bge.py.

    bge models require normalized embeddings for cosine similarity; the
    universal HuggingFaceEmbeddings class provides that via encode_kwargs.
    """

    def __init__(self, model_name: str = "BAAI/bge-base-en-v1.5"):
        self.model = HuggingFaceEmbeddings(
            model_name=model_name,
            encode_kwargs={"normalize_embeddings": True},
        )

    def embed_query(self, text: str) -> list[float]:
        return self.model.embed_query(text)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.embed_documents(texts)


class E5Embedding(Embeddings):
    """Inline E5 wrapper — mirrors src/embeddings/e5.py.

    E5 models are trained with instruction prefixes: queries are prefixed
    "query: " and passages "passage: " before embedding. Unlike BGE, the
    output vectors are NOT normalized here.
    """

    def __init__(self, model_name: str = "intfloat/multilingual-e5-base"):
        self.model = HuggingFaceEmbeddings(model_name=model_name)

    def embed_query(self, text: str) -> list[float]:
        return self.model.embed_query(f"query: {text}")

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.embed_documents([f"passage: {text}" for text in texts])


# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `NUM_PASSAGES = 400` is the embedding budget (first 400 passages of the corpus); `NUM_QUESTIONS = 60` is the question pool (first 60 test questions with non-empty answers); `PREVIEW` controls how many characters the demo shows around the answer; `MODEL_NAME` names the local BGE model; and the two parquet paths point at the rag-mini-wikipedia files already on disk.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the comparison
# --------------------------------------------------------------------------
PASSAGES_PARQUET = "Data/corpus/rag-mini-wikipedia/passages.parquet"
TEST_PARQUET = "Data/corpus/rag-mini-wikipedia/test.parquet"
MODEL_NAME = "BAAI/bge-base-en-v1.5"
E5_MODEL_NAME = "intfloat/multilingual-e5-base"
NUM_PASSAGES = 400  # first N passages of the corpus (embedding budget)
NUM_QUESTIONS = 60  # first N test questions with non-empty answers
PREVIEW = 170  # max characters shown around the answer in the example table


## 2. Load — deterministic subsets of the corpus

`load_subsets` returns `(passages, qa_pairs)` with zero randomness: the first `NUM_PASSAGES` rows of `passages.parquet`, and the first `NUM_QUESTIONS` rows of `test.parquet` whose answer is a non-empty string, in file order.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — deterministic subsets of the corpus
# --------------------------------------------------------------------------
def load_subsets() -> tuple[list[str], list[tuple[str, str]]]:
    """Return (passages, qa_pairs) with zero randomness.

    Passages: first ``NUM_PASSAGES`` rows of ``passages.parquet``. Questions:
    first ``NUM_QUESTIONS`` rows of ``test.parquet`` whose answer is a
    non-empty string, in file order.
    """
    passages_df = pd.read_parquet(PASSAGES_PARQUET)
    passages = [str(p) for p in passages_df["passage"].head(NUM_PASSAGES).tolist()]

    test_df = pd.read_parquet(TEST_PARQUET)
    qa_pairs: list[tuple[str, str]] = []
    for question, answer in zip(test_df["question"], test_df["answer"]):
        if isinstance(answer, str) and answer.strip():
            qa_pairs.append((str(question), answer.strip()))
        if len(qa_pairs) >= NUM_QUESTIONS:
            break
    return passages, qa_pairs


## 3. Score — embed once per model, cosine via normalized dot products

`l2_normalize` row-normalizes a matrix to unit length. `score_matrix` embeds all passages (batched) and questions (one at a time, `embed_query`), then returns the `(n_questions, n_passages)` cosine-similarity matrix — both sides L2-normalized first, so (a) dot product equals cosine similarity and (b) the mean/median similarity reported for each model lives on the same unit-scale. `contains_answer` is the answer-containment test. `ModelScore` packs the per-question retrieval outcomes of one model (top-1 index, top-1 similarity, hit/miss per question) with its recall/mean/median accessors. `escape` makes newlines visible, `passage_window` previews the passage centered on the answer span, and `disagreement_indices` finds the question indices where the two models disagree on the top-1 hit.


In [ ]:
# --------------------------------------------------------------------------
# 3. Score — embed once per model, cosine via normalized dot products
# --------------------------------------------------------------------------
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    """Row-wise L2 normalization to unit length."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)


def score_matrix(embedder: object, passages: list[str],
                 questions: list[str]) -> np.ndarray:
    """Return the (n_questions, n_passages) cosine-similarity matrix.

    E5's wrapper does not normalize its output (unlike BGE's); both matrices
    are L2-normalized here so that (a) dot product equals cosine similarity
    and (b) the mean/median similarity reported for each model lives on the
    same unit-scale.
    """
    passage_vecs = np.asarray(embedder.embed_documents(passages), dtype=np.float32)
    query_vecs = np.asarray([embedder.embed_query(q) for q in questions],
                            dtype=np.float32)
    return l2_normalize(query_vecs) @ l2_normalize(passage_vecs).T


def contains_answer(passage: str, answer: str) -> bool:
    """Answer containment: case-insensitive substring on stripped strings."""
    return answer.lower() in passage.lower()


class ModelScore:
    """Per-question retrieval outcomes for one embedding model."""

    def __init__(self, name: str, cosine: np.ndarray,
                 passages: list[str], qa_pairs: list[tuple[str, str]]) -> None:
        self.name = name
        self.cosine = cosine
        self.top1_idx = cosine.argmax(axis=1)
        rows = np.arange(cosine.shape[0])
        self.top1_sim = cosine[rows, self.top1_idx]
        self.hits = [contains_answer(passages[i], answer)
                     for i, (_, answer) in zip(self.top1_idx, qa_pairs)]

    def recall_at_1(self) -> float:
        return float(np.mean(self.hits))

    def mean_sim(self) -> float:
        return float(np.mean(self.top1_sim))

    def median_sim(self) -> float:
        return float(np.median(self.top1_sim))


def escape(s: str) -> str:
    """Make newlines visible so passage previews stay on one line."""
    return s.replace("\n", "\\n")


def passage_window(passage: str, answer: str, width: int = PREVIEW) -> str:
    """Preview the passage, centered on the answer span when it is present."""
    idx = passage.lower().find(answer.lower())
    if idx < 0:
        return escape(passage[:width])
    start = max(0, idx - width // 3)
    end = min(len(passage), idx + len(answer) + width // 2)
    preview = escape(passage[start:end])
    prefix = "..." if start > 0 else ""
    suffix = "..." if end < len(passage) else ""
    return f"{prefix}{preview}{suffix}"


def disagreement_indices(bge: ModelScore, e5: ModelScore, limit: int) -> list[int]:
    """Question indices where the two models disagree on the top-1 hit."""
    return [i for i, (hb, he) in enumerate(zip(bge.hits, e5.hits))
            if hb != he][:limit]


## 4. Run the experiment — embed with both models, score, collect

`run_experiment` loads the subsets, builds both inline embedders, probes the embedding dimension of each, scores both models over the 400 passages and 60 questions (timed), and returns everything the demo and the gate need — no printing happens here.


In [ ]:
# --------------------------------------------------------------------------
# 4. Run the experiment — embed with both models, score, collect
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passages, qa_pairs = load_subsets()
    questions = [q for q, _ in qa_pairs]

    models = [("BGE", BGEEmbedding(model_name=MODEL_NAME)),
              ("E5 ", E5Embedding(model_name=E5_MODEL_NAME))]
    dims: dict[str, int] = {}
    for label, embedder in models:
        probe = embedder.embed_query(questions[0])
        dims[label.strip()] = len(probe)

    results: dict[str, ModelScore] = {}
    timings: dict[str, float] = {}
    for label, embedder in models:
        t0 = time.perf_counter()
        cosine = score_matrix(embedder, passages, questions)
        elapsed = time.perf_counter() - t0
        results[label.strip()] = ModelScore(label.strip(), cosine,
                                            passages, qa_pairs)
        timings[label.strip()] = elapsed

    return {
        "passages": passages,
        "qa_pairs": qa_pairs,
        "dims": dims,
        "results": results,
        "timings": timings,
    }


## 5. Demo — print the artifact

`print_demo(exp)` prints the artifact from five angles: the setup (subsets, both models with dims and their caveats); the per-model embedding timings; the recall@1 table; the mean/median top-1 similarity table; the side-by-side disagreement examples (up to 3 questions where the models pick different top-1 passages, with the gold answer and the retrieved snippet); and a takeaway on the winner, why absolute similarity does not transfer across models, and why E5 needs its prefixes plus L2 normalization to be compared fairly at all.


In [ ]:
# --------------------------------------------------------------------------
# 5. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    passages, qa_pairs = exp["passages"], exp["qa_pairs"]
    results, dims, timings = exp["results"], exp["dims"], exp["timings"]
    bge, e5 = results["BGE"], results["E5"]

    print("=" * 66)
    print("Lab 03 — BGE vs E5: head-to-head retrieval comparison")
    print("=" * 66)

    print(f"\n[1] Setup")
    print(f"    passages : {len(passages)} (first {NUM_PASSAGES} of rag-mini-wikipedia)")
    print(f"    questions: {len(qa_pairs)} (first {NUM_QUESTIONS} with non-empty answers)")
    print(f"    BGE : BAAI/bge-base-en-v1.5            dim={dims['BGE']}  "
          f"(wrapper normalizes output)")
    print(f"    E5  : intfloat/multilingual-e5-base   dim={dims['E5']}  "
          f"(wrapper auto-prefixes 'query:'/'passage:', no normalization)")
    print("    both matrices L2-normalized here so cosine is a fair dot product")

    for name in ("BGE", "E5"):
        print(f"    embedded {len(passages)} passages + {len(qa_pairs)} queries "
              f"with {name} in {timings[name]:.1f}s")

    print("\n[2] Recall@1 — gold answer found in the top-1 passage")
    print(f"    {'model':<10}{'recall@1':>10}{'hits':>7}{'misses':>9}")
    for score in (bge, e5):
        hits = sum(score.hits)
        print(f"    {score.name:<10}{score.recall_at_1():>10.3f}"
              f"{hits:>7}{len(score.hits) - hits:>9}")

    print("\n[3] Similarity of the retrieved passage (top-1 cosine)")
    print(f"    {'model':<10}{'mean':>8}{'median':>9}")
    for score in (bge, e5):
        print(f"    {score.name:<10}{score.mean_sim():>8.3f}"
              f"{score.median_sim():>9.3f}")

    print("\n[4] Example retrievals where the models disagree")
    picked = disagreement_indices(bge, e5, limit=3)
    if not picked:
        print("    No disagreement found in this subset — both models agree "
              "on every question.")
    for i in picked:
        question, answer = qa_pairs[i]
        print(f"\n    Q: {escape(question[:110])}")
        print(f"       gold answer: {answer!r}")
        for score in (bge, e5):
            verdict = "HIT " if score.hits[i] else "MISS"
            snippet = passage_window(passages[score.top1_idx[i]], answer)
            print(f"       {score.name:<4}{verdict} sim={score.top1_sim[i]:.3f}  "
                  f"{snippet[:PREVIEW + 8]}{'...' if len(snippet) > PREVIEW + 8 else ''}")

    print("\n[5] Takeaway")
    winner = "BGE" if bge.recall_at_1() > e5.recall_at_1() else \
        "E5" if e5.recall_at_1() > bge.recall_at_1() else "neither"
    print(f"    On {len(qa_pairs)} real questions over {len(passages)} passages, "
          f"{winner} wins recall@1 "
          f"(BGE {bge.recall_at_1():.3f} vs E5 {e5.recall_at_1():.3f}).")
    print("    Recall@1 only measures whether the top passage *contains* the")
    print("    gold answer — a coarse proxy (many answers here are 'yes'/'no').")
    print("    E5's top-1 similarities sit higher (median "
          f"{e5.median_sim():.3f} vs {bge.median_sim():.3f}) even with both")
    print("    matrices unit-length: absolute similarity does NOT transfer")
    print("    across models, only rankings do. And E5 needs its 'query:'/")
    print("    'passage:' prefixes plus L2 normalization to be compared")
    print("    fairly against BGE at all.")


## 6. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `NUM_PASSAGES` passages and `NUM_QUESTIONS` Q/A pairs loaded; both models produce 768-dim vectors; every `ModelScore` holds `NUM_QUESTIONS` per-question outcomes with recall@1 and top-1 similarities in `[-1, 1]`; each model's reported recall/mean/median are self-consistent with its stored arrays; and the cosine matrices have the expected `(NUM_QUESTIONS, NUM_PASSAGES)` shape. Every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 6. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append((f"exactly {NUM_PASSAGES} passages loaded",
                   len(exp["passages"]) == NUM_PASSAGES))
    checks.append((f"exactly {NUM_QUESTIONS} questions with non-empty answers",
                   len(exp["qa_pairs"]) == NUM_QUESTIONS))
    checks.append((f"BGE dim = {exp['dims']['BGE']} (768)", exp["dims"]["BGE"] == 768))
    checks.append((f"E5 dim = {exp['dims']['E5']} (768)", exp["dims"]["E5"] == 768))

    for name in ("BGE", "E5"):
        ms = exp["results"][name]
        checks.append((f"{name}: {NUM_QUESTIONS} per-question outcomes",
                       len(ms.hits) == NUM_QUESTIONS))
        checks.append((f"{name}: cosine matrix is {NUM_QUESTIONS} x {NUM_PASSAGES}",
                       ms.cosine.shape == (NUM_QUESTIONS, NUM_PASSAGES)))
        checks.append((f"{name}: recall@1 in [0, 1] (got {ms.recall_at_1():.3f})",
                       0.0 <= ms.recall_at_1() <= 1.0))
        checks.append((f"{name}: top-1 similarity in [-1, 1] "
                       f"(mean {ms.mean_sim():.3f})",
                       -1.0 <= ms.mean_sim() <= 1.0))
        recomputed = float(np.mean(ms.hits))
        checks.append((f"{name}: recall@1 self-consistent (recomputed "
                       f"{recomputed:.3f})", abs(recomputed - ms.recall_at_1()) < 1e-9))
        recomputed_mean = float(np.mean(ms.top1_sim))
        checks.append((f"{name}: mean top-1 similarity self-consistent "
                       f"(recomputed {recomputed_mean:.3f})",
                       abs(recomputed_mean - ms.mean_sim()) < 1e-9))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few minutes of local embedding (BGE + E5, both cached on disk) — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The head-to-head tables: recall@1, top-1 similarity, and the side-by-side retrievals where the two models disagree — with the gold answer and the retrieved snippet for each.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
